In [1]:
# ==============================================================================
# CÉLULA 1: SETUP E CRIAÇÃO DO DATASET
# ==============================================================================

import pandas as pd
import numpy as np
import torch
import os

# ----------------------------------------
# 1. Configurações
# ----------------------------------------
N_DIAS_HISTORICO = 100
HORIZONTE_PREVISAO = 14 # Prever os próximos 14 dias
FREQ = "D" # Frequência diária
DATA_DIR = "../../data" # Diretório para salvar os dados

# Criar o diretório se não existir
os.makedirs(DATA_DIR, exist_ok=True)

# ----------------------------------------
# 2. Criar a Série Temporal Principal (target)
# ----------------------------------------
datas_hist = pd.date_range(start="2023-01-01", periods=N_DIAS_HISTORICO, freq=FREQ)

# Criar dados sintéticos com tendência e sazonalidade semanal
trend = np.linspace(50, 100, N_DIAS_HISTORICO)
seasonality = 15 * np.sin(np.arange(N_DIAS_HISTORICO) * 2 * np.pi / 7)
noise = np.random.normal(0, 2, N_DIAS_HISTORICO)
target = trend + seasonality + noise

# ----------------------------------------
# 3. Criar Covariáveis (Conhecidas no Futuro)
# ----------------------------------------
day_of_week_hist = datas_hist.dayofweek
month_hist = datas_hist.month

# ----------------------------------------
# 4. Criar DataFrame Histórico
# ----------------------------------------
df_hist = pd.DataFrame({
'date': datas_hist,
'target': target,
'day_of_week': day_of_week_hist,
'month': month_hist
}).set_index('date')

print("--- DataFrame Histórico (df_hist) ---")
print(df_hist.tail())


# ----------------------------------------
# 5. Criar DataFrame Futuro (Apenas para Covariáveis)
# ----------------------------------------
datas_fut = pd.date_range(start=df_hist.index[-1] + pd.Timedelta(days=1), periods=HORIZONTE_PREVISAO, freq=FREQ)

day_of_week_fut = datas_fut.dayofweek
month_fut = datas_fut.month

df_future_covariates = pd.DataFrame({
'date': datas_fut,
'day_of_week': day_of_week_fut,
'month': month_fut
}).set_index('date')

print("\n--- DataFrame de Covariáveis Futuras (df_future_covariates) ---")
print(df_future_covariates.head())

# ----------------------------------------
# 6. Definir dispositivo (GPU ou CPU)
# ----------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsando dispositivo: {device}")

# ----------------------------------------
# 7. Salvar os DataFrames em arquivos Parquet
# ----------------------------------------
hist_path = os.path.join(DATA_DIR, "hist.parquet")
future_cov_path = os.path.join(DATA_DIR, "future_covariates.parquet")

df_hist.to_parquet(hist_path)
df_future_covariates.to_parquet(future_cov_path)

print(f"\nDataFrame histórico salvo em: {hist_path}")
print(f"DataFrame de covariáveis futuras salvo em: {future_cov_path}")

--- DataFrame Histórico (df_hist) ---
                target  day_of_week  month
date                                      
2023-04-06   91.026088            3      4
2023-04-07   82.740630            4      4
2023-04-08   87.401396            5      4
2023-04-09  100.150267            6      4
2023-04-10  112.014606            0      4

--- DataFrame de Covariáveis Futuras (df_future_covariates) ---
            day_of_week  month
date                          
2023-04-11            1      4
2023-04-12            2      4
2023-04-13            3      4
2023-04-14            4      4
2023-04-15            5      4

Usando dispositivo: cpu

DataFrame histórico salvo em: ../../data/hist.parquet
DataFrame de covariáveis futuras salvo em: ../../data/future_covariates.parquet
